# HW3P2: Automatic Speech Recognition with an Encoder-Decoder Transformer in JAX


# Schedule:
- Checkpoint Submission (Friday, October 30, 2026 @ 11:59PM ET)
- Kaggle Submission (Friday, November 6, 2026 @ 11:59PM ET | Slack Deadline is Monday, November 16, 2026 @ 11:59PM ET)
- Code Submission  (Sunday, November 8, 2026 @ 11:59PM ET)


## Requirement Acknowledgement

Setting the flag below to **True** indicates that you have read, understood, and agree to comply with **all** of the following requirements:

1. **Slack Days Usage**  
   Slack days may be used **only** for the **P2 Final submission**, *not* for the checkpoint.  
   Specifically, slack days may be applied to submit **final P2 Kaggle scores** to the **Slack Kaggle Competition**, at the expense of your available slack days.

2. **Final Code Submission Deadline**  
   The final **Gradescope code submission** is due **within 48 hours after** the Kaggle competition deadline,  
   or **on the same day** as your final Kaggle submission—whichever occurs first.

3. **Kaggle Username & Score Verification**  
   You must provide your **Kaggle username** below. We will use it to retrieve your official **PRIVATE leaderboard** score.  
   Minor score or output variance is acceptable; however, any discrepancies caused by **modifications to the required submission code**
   (including the notebook’s final submission cell) will result in an **Academic Integrity Violation (AIV)**.

4. **Model Implementation Restrictions**  
   You are **not permitted** to use pre-trained or pre-loaded models (e.g., from Hugging Face or similar libraries).  
   You **may** implement models described in research papers or articles, but they **must** be implemented **from scratch**
   using fundamental JAX / Flax NNX components (e.g., `nnx.Linear`, `nnx.Conv`, etc.).

5. **Data Usage Restrictions**  
   The use of **any external data or datasets** is strictly prohibited at **any stage** of this assignment.

6. **Collaboration Policy**  
   You may collaborate with teammates to run experiments or ablations.  
   However, you must submit **your own code** and **your own results**.

7. **Academic Integrity**  
   Failure to comply with any of the above requirements will be treated as an **Academic Integrity Violation (AIV)**.

8. **Late Submissions**  
   Late submissions **must** be submitted through the **Slack Kaggle Competition** (see write-up for details).  
   Submissions made to the **regular Kaggle competition** after the original deadline will **not** be considered,
   regardless of remaining slack days.


In [ ]:
ACKNOWLEDGED = False #TODO: Only set Acknowledged to True if you have read the above acknowlegements and agree to ALL of them.

# Setup
-  Follow the setup instructions based on your preferred environment!

## Local

One of our key goals in designing this assignment is to allow you to complete most of the preliminary implementation work locally.  
We highly recommend that you **pass all tests locally** using the provided `hw3_data_subset` before moving to a GPU runtime.  
To do this, simply:

### Create a new conda environment
```bash
# Be sure to deactivate any active environments first
conda create -n hw3 python=3.12.4
```

### Activate the conda environment
```bash
conda activate hw3
```

### Install the dependencies using the provided `requirements.txt`
```bash
pip install --no-cache-dir --ignore-installed -r requirements.txt
```

### Ensure that your notebook is in the same working directory as the `Handout`
This can be achieved by:
1. Physically moving the notebook into the handout directory.
2. Changing the notebook’s current working directory to the handout directory using the os.chdir() function.

### Open the notebook and select the newly created environment from the kernel selector.

If everything was done correctly, You should see atleast the following files in your current working directory after running `!ls`:
```
.
├── README.md
├── requirements.txt
├── hw3lib/
├── tests/
└── hw3_data_subset/
```

## Colab

### Step 1: Get your handout
- See writeup for recommended approaches.

In [ ]:
# Example: My preferred approach
import os
# Settings -> Developer Settings -> Personal Access Tokens -> Token (classic)
os.environ['GITHUB_TOKEN'] = "your_github_token_here"

GITHUB_USERNAME = "your_github_username_here"
REPO_NAME       = "your_github_repo_name_here"
TOKEN = os.environ.get("GITHUB_TOKEN")
repo_url        = f"https://{TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
!git clone {repo_url}

In [ ]:
# To pull latest changes (Must be in the repo dir, use pwd/ls to verify)
!cd {REPO_NAME} && git pull

### Step 2: Obtain Data

- `NOTE`: This process will automatically download and unzip data for both `HW3P1` and `HW3P2`.  


In [ ]:
!curl -L -o /content/TODO-F26-HW3-DATA.zip https://www.kaggle.com/api/v1/datasets/download/cmu11785/TODO-F26-HW3-DATA
!unzip -q -o /content/TODO-F26-HW3-DATA.zip -d /content/hw3_data
!rm -rf /content/TODO-F26-HW3-DATA.zip
!du -h --max-depth=2 /content/hw3_data

### Step 3: Set up Kaggle API Authentication

Join the HW3P2 kaggle competition (https://www.kaggle.com/t/9a00ec2986e64408b581250f8a401880)

In [ ]:
import os
os.environ['KAGGLE_USERNAME'] = "<your-username>" # TODO: Verify in Settings
os.environ['KAGGLE_API_TOKEN'] = "<your-key>" # TODO: Add Access Token

# Verify
import kaggle
api = kaggle.api  # Already authenticated on import

# Verify authentication works
api.competitions_list_cli()
api.competition_download_files('hw-3-p-2-f-26-sandbox')

# Install dependencies for JAX HW3P2
!pip install -q jax jaxlib flax optax pyyaml Levenshtein wandb tqdm pandas matplotlib


In [ ]:
!pwd
%cd /content/

In [ ]:
%pip install --no-deps -r {REPO_NAME}/requirements.txt
import os
os.kill(os.getpid(), 9) # NOTE: This will restart the your colab Python runtime (required)!

In [ ]:
!pip install transformers -U

### Step 5: Move to Handout Directory
You must be within the handout directory for the library imports to work!

- `NOTE`: You may have to repeat running this command anytime you restart your runtime.
- `NOTE`: You can do a `pwd` to check if you are in the right directory.
- `NOTE`: The way it is setup currently, Your data directory should be one level up from your project directory. Keep this in mind when you are setting your `root` in the config file.

If everything was done correctly, You should see atleast the following files in your current working directory after running `!ls`:
```
.
├── README.md
├── requirements.txt
├── hw3lib/
├── tests/
└── hw3_data_subset/

```

In [ ]:
import os
REPO_NAME = "your_github_repo_name_here"  # same name as in the clone cell; re-set after every runtime restart
os.chdir(REPO_NAME)
!ls

## PSC

### 1️⃣ **Step 1 Setting Up Your Environment on Bridges2**

❗️⚠️ For this homework, we are **providing a shared Conda environment** for the entire class. Therefore, PSC users **do not need to manually install any packages**.

❗️⚠️ For this homework, you need to **download the dataset to the node `$LOCAL`** to avoid I/O bottlenecks from the shared filesystem. This means that each time you run on a new node, you need to download the dataset again. However, as long as you stay on the same node, you do not need to re-download the dataset. Please refer to **Step 3** for the detailed procedure.

Follow these steps to set up the environment and start a Jupyter notebook on Bridges2:

- To run your notebook more efficiently on PSC, we need to use a **Jupyter Server** hosted on a compute node.

- You can use your prefered way of connecting to the Jupyter Server.  **The recommended way is to connect in VSCode.**
Follow the instructions below.


**The recommended way of connecting is:**

#### **Connect in VSCode**
SSH into Bridges2 and navigate to your **Jet directory** (`Jet/home/<your_psc_username>`). Upload your notebook there, and then connect to the Jupyter Server from that directory.

#### **1. SSH into Bridges2**
1）Open VS Code and click on the `Extensions` icon in the left sidebar. Make sure the "**Remote - SSH**" extension is installed.

2）Open the command palette (**Shift+Command+P** on Mac, **Ctrl+Shift+P** on Windows). A search box will appear at the top center. Choose `"Remote-SSH: Add New SSH Host"`, then enter:

```bash
ssh <your_username>@bridges2.psc.edu #change <your_username> to your username
```

Next, choose `"/Users/<your_username>/.ssh/config"` as the config file. A dialog will appear in the bottom right saying "Host Added". Click `"Connect"`, and then enter your password.

(Note: After adding the host once, you can later use `"Remote-SSH: Connect to Host"` and select "bridges2.psc.edu" from the list.)

3）Once connected, click `"Explorer"` in the left sidebar > "Open Folder", and navigate to your home directory under the project grant:
```bash
/jet/home/<your_username>  #change <your_username> to your username
```

4）You can now drag your notebook files directly into the right-hand pane (your remote home directory), or upload them using `scp` into your folder.

> ❗️⚠️ The following steps should be executed in the **VSCode integrated terminal**.

#### **2. Navigate to Your Directory**
Make sure to use this `/jet/home/<your_username>` as your working directory, since all subsequent operations (up to submission) are based on this path.
```bash
cd /jet/home/<your_username>  #change <your_username> to your username
```

#### **3. Request a Compute Node**
```bash
interact -p GPU-shared --gres=gpu:v100-32:1 -t 8:00:00 -A cis250019p
```

#### **4. Load the Anaconda Module**
```bash
module load anaconda3
```

#### **5. Activate the provided HW3 Environment**

```bash
conda deactivate # First, deactivate any existing Conda environment
conda activate /ocean/projects/cis250019p/mzhang23/TA/envs/IDLS26 && export PYTHONNOUSERSITE=1

#### **6. Start Jupyter Notebook**
Launch Jupyter Notebook:
```bash
jupyter notebook --no-browser --ip=0.0.0.0
```

Go to **Kernel** → **Select Another Kernel** → **Existing Jupyter Server**
   Enter the URL of the Jupyter Server:```http://{hostname}:{port}/tree?token={token}```
   
   *(Usually, this URL appears in the terminal output after you run `jupyter notebook --no-browser --ip=0.0.0.0`, in a line like:  “Jupyter Server is running at: http://...”)*

   - eg: `http://v011.ib.bridges2.psc.edu:8888/tree?token=e4b302434e68990f28bc2b4ae8d216eb87eecb7090526249`

> **Note**: Replace `{hostname}`, `{port}` and `{token}` with your actual values from the Jupyter output.

After launching the Jupyter notebook, you can run the cells directly inside the notebook — no need to use the terminal for the remaining steps.

### 2️⃣ Step 2: Get Repo

In [ ]:
#Make sure you are in your directory
!pwd #should be /jet/home/<your_username>, if not, uncomment the following line and replace with your actual username:
%cd /jet/home/<your_username>
#TODO: replace the "<your_username>" to yours

In [ ]:
# Example: My preferred approach
import os
# Settings -> Developer Settings -> Personal Access Tokens -> Token (classic)
os.environ['GITHUB_TOKEN'] = "your_github_token_here"

GITHUB_USERNAME = "your_github_username_here"
REPO_NAME       = "your_github_repo_name_here"
TOKEN = os.environ.get("GITHUB_TOKEN")
repo_url        = f"https://{TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
!git clone {repo_url}

In [ ]:
# To pull latest changes (Must be in the repo dir, use pwd/ls to verify)
!cd {REPO_NAME} && git pull

#### **Move to Project Directory**
- `NOTE`: You may have to repeat this on anytime you restart your runtime. You can do a `pwd` or `ls` to check if you are in the right directory.

In [ ]:
import os
REPO_NAME = "your_github_repo_name_here"  # same name as in the clone cell; re-set after every runtime restart
os.chdir(REPO_NAME)
!ls

### 3️⃣ **Step 3: Set up Kaggle API Authentication**

Join the HW3P2 kaggle competition (https://www.kaggle.com/t/9a00ec2986e64408b581250f8a401880)

In [ ]:
import os
os.environ['KAGGLE_USERNAME'] = "<your-username>" # TODO: Verify in Settings
os.environ['KAGGLE_API_TOKEN'] = "<your-key>" # TODO: Add Access Token

# Verify
import kaggle
api = kaggle.api  # Already authenticated on import

# Verify authentication works
api.competitions_list_cli()
api.competition_download_files('hw-3-p-2-f-26-sandbox')

### 4️⃣ **Step 4: Get Data**

❗️⚠️ In this homework, you need to download the dataset to the **GPU node’s local storage (`$LOCAL`)** instead of using the shared /ocean directory, in order to avoid I/O bottlenecks. Using the shared filesystem may slow down training drastically and can take hours per epoch.

Note that **the local storage on a compute node is temporary and will be cleared** when your node time limit is reached or when you move to a different node. Therefore, **every time you run on a new node, you need to re-run the dataset download step**. However, as long as you stay on the same node, you do NOT need to download the dataset again.

In [ ]:
!mkdir -p $LOCAL/dataset
api.dataset_download_files(
    "cmu11785/hw-3-p-2-f-26-sandbox",
    path=f"{os.environ['LOCAL']}/dataset",
    unzip=True
)

In [ ]:
import os
data_path = "/local/dataset/hw3p2_data" # this is the path of the dataset on your node
print("Files in shared hw3p2 dataset:", os.listdir(data_path))

In [ ]:
!apt-get install tree
!tree -L 2 /local/dataset/hw3p2_data

# Imports
- If your setup was done correctly, you should be able to run the following cell without any issues.

In [ ]:
from hw3lib.data import (
    H3Tokenizer,
    ASRDataset,
    verify_dataloader
)
from hw3lib.model import (
    EncoderDecoderTransformer
)
from hw3lib.utils import (
    create_scheduler,
    create_optimizer,
    plot_lr_schedule
)
from hw3lib.trainers import (
    ASRTrainer
)
import jax
import jax.numpy as jnp
from flax import nnx
import optax
from torch.utils.data import DataLoader
import yaml
import gc
import os
import json
import wandb
import pandas as pd
import matplotlib.pyplot as plt

print(f"JAX Default Backend: {jax.default_backend()}")
print(f"JAX Devices: {jax.devices()}")


# Implementations
- `NOTE`: All of these implementations have detailed specification, implementation details, and hints in their respective source files. Make sure to read all of them in their entirety to understand the implementation details!

## Dataset Implementation
- Implement the `ASRDataset` class in `hw3lib/data/asr_dataset.py`.
- You will have to implement parts of `__init__` and completely implement the `__len__`, `__getitem__` and `collate_fn` methods.
- Run the cell below to check your implementation.


In [ ]:
!python -m tests.test_dataset_asr

## Model Implementations

Overview:

- Implement the `PadMask` and `CausalMask` functions in `hw3lib/model/masks.py`.
- Implement the `PositionalEncoding` and `RotaryPositionalEncoding` classes in `hw3lib/model/positional_encoding.py`.
- Implement the `RMSNorm` class in `hw3lib/model/norms.py`.
- Implement the `MultiHeadAttention` class in `hw3lib/model/attention.py`.
- Implement the `SelfAttentionLayer`, `CrossAttentionLayer`, `SwiGLU` and `FeedForwardLayer` classes in `hw3lib/model/sublayers.py`.
- Implement the `CrossAttentionDecoderLayer` class in `hw3lib/model/decoder_layers.py`.
- Implement the `SelfAttentionEncoderLayer` class in `hw3lib/model/encoder_layers.py`. This will be mostly a copy-paste of the `CrossAttentionDecoderLayer`, without the cross-attention sublayer and without the causal constraint on the input sequence.
- Implement the `EncoderDecoderTransformer` class in `hw3lib/model/transformers.py`.

Implement them in the order below. Each one is used by the next.

### Masks
- Implement the `PadMask` and `CausalMask` functions in `hw3lib/model/masks.py`.
- Run the cells below to check your implementations.

In [ ]:
!python -m tests.test_mask_padding

In [ ]:
!python -m tests.test_mask_causal

### Positional Encoding
- Implement the `PositionalEncoding` class in `hw3lib/model/positional_encoding.py`.
- This is the fixed sinusoidal encoding, added to the embeddings before the first layer.
- Run the cell below to check your implementation.

In [ ]:
!python -m tests.test_positional_encoding

### Rotary Positional Encoding
- Implement the `RotaryPositionalEncoding` class in `hw3lib/model/positional_encoding.py`.
- Unlike `PositionalEncoding`, this one is not added to the embeddings. It rotates the queries and keys inside each attention operation, so it is applied in `MultiHeadAttention` rather than at the top of the model.
- Run the cell below to check your implementation.

In [ ]:
!python -m tests.test_rope

### Normalization
- Implement the `RMSNorm` class in `hw3lib/model/norms.py`.
- Run the cell below to check your implementation.

In [ ]:
!python -m tests.test_norms

### Multi-Head Attention
- Implement the `MultiHeadAttention` class in `hw3lib/model/attention.py`.
- You implemented this in NumPy for `HW3P1`, in `mytorch/nn/multi_head_attention.py`. This is the same module in JAX and Flax NNX, so keep that file open beside this one. The method names and the `forward` signature are unchanged, `backward` and `init_weights` are no longer needed, and rotary embeddings and QK-Norm are new.
- Run the cell below to check your implementation.

In [ ]:
!python -m tests.test_attention

### Transformer Sublayers
- Implement the `SelfAttentionLayer`, `CrossAttentionLayer`, `SwiGLU` and `FeedForwardLayer` classes in `hw3lib/model/sublayers.py`.
- `NOTE`: `SwiGLU` and `FeedForwardLayer` are separate. `FeedForwardLayer` selects between `SwiGLU` and the GELU network based on `ffn_type`.
- Run the cells below to check your implementations.

In [ ]:
!python -m tests.test_sublayer_selfattention

In [ ]:
!python -m tests.test_sublayer_feedforward

In [ ]:
!python -m tests.test_sublayer_crossattention

### Transformer Cross-Attention Decoder Layer
- Implement the `CrossAttentionDecoderLayer` class in `hw3lib/model/decoder_layers.py`.
- Then run the cell below to check your implementation.


In [ ]:
!python -m tests.test_decoderlayer_crossattention

### Transformer Self-Attention Encoder Layer
- Implement the `SelfAttentionEncoderLayer` class in `hw3lib/model/encoder_layers.py`.
- Then run the cell below to check your implementation.




In [ ]:
!python -m tests.test_encoderlayer_selfattention

### Encoder-Decoder Transformer

- Implement the  `EncoderDecoderTransformer` class in `hw3lib/model/transformers.py`.
- Then run the cell below to check your implementation.

In [ ]:
!python -m tests.test_transformer_encoder_decoder

## Decoding Implementation
- Implement the `generate_greedy` method of the `SequenceGenerator` class in `hw3lib/decoding/sequence_generator.py`.
- Greedy search is required. The trainer's `recognize` method uses it by default, and so does the inference cell at the end of this notebook.
- Run the cell below to check your implementation.

In [ ]:
!python -m tests.test_decoding --mode greedy

### Beam Search
- We highly recommend you to implement the `generate_beam` method of the `SequenceGenerator` class in `hw3lib/decoding/sequence_generator.py`.
- Then run the cell below to check your implementation.
- `NOTE`: This is an optional but highly recommended task for `HW3P2` to ease the journey to high cutoffs!

In [ ]:
!python -m tests.test_decoding --mode beam

## Trainer Implementation
You will have to do some minor in-filling for the `ASRTrainer` class in `hw3lib/trainers/asr_trainer.py` before you can use it.
- Fill in the `TODO`s in the `ctc_and_ce_loss`.
- Fill in the `TODO`s in the `train_step`.
- Fill in the `TODO`s in the `recognize` method.

`WARNING`: There are no test's for this. Implement carefully!

# Experiments
From this point onwards you may want to switch to a `GPU` runtime.
- `OBJECTIVE`: Optimize your model for `CER` on the test set.

## Config
- You can use the `config.yaml` file to set your config for your ablation study.

---
### Notes:

- Set `tokenization: token_type:` to specify your desired tokenization strategy
- You will need to set the root path to your `hw3p2_data` folder in `data: root:`. This will depend on your setup. For eg.
  - `Colab:`: `"/content/hw3_data/hw3p2_data"`
  - `PSC`: `"/local/dataset/hw3p2_data"`
- There's extra configurations in the `optimizer` section which will only be relevant if you decide to use the `create_optimizer` function we've provided in `hw3lib/utils/create_optimizer.py`.
- `BE CAREFUL` while setting numeric values. Eg. `1e-4` will get serialized to a `str` while `1.0e-4` gets serialized to float.

In [ ]:
%%writefile config.yaml

Name                      : "Baseline-JAX-ASR"

###### Tokenization ------------------------------------------------------------
tokenization:
  token_type                : "5k"       # [char, 1k, 5k, 10k]
  token_map :
      'char': 'hw3lib/data/tokenizer_jsons/tokenizer_char.json'
      '1k'  : 'hw3lib/data/tokenizer_jsons/tokenizer_1000.json'
      '5k'  : 'hw3lib/data/tokenizer_jsons/tokenizer_5000.json'
      '10k' : 'hw3lib/data/tokenizer_jsons/tokenizer_10000.json'

###### Dataset -----------------------------------------------------------------
data:
  root                 : "hw3_data/hw3p2_data"  # Root path of your data
  train_partition      : "train-clean-100"  # paired text-speech for ASR pre-training
  val_partition        : "dev-clean"        # paired text-speech for ASR pre-training
  test_partition       : "test-clean"       # paired text-speech for ASR pre-training
  subset               : 1.0                # Load a subset of the data (for debugging, testing, etc.)
  batch_size           : 16           
  NUM_WORKERS          : 2            # Set to 0 for CPU
  norm                 : 'global_mvn' # ['global_mvn', 'cepstral', 'none']
  num_feats            : 80

  ###### SpecAugment ---------------------------------------------------------------
  specaug                   : False  # Set to True if you want to use SpecAugment
  specaug_conf:
    apply_freq_mask         : True
    freq_mask_width_range   : 5
    num_freq_mask           : 2
    apply_time_mask         : True
    time_mask_width_range   : 40
    num_time_mask           : 2

###### Network Specs -------------------------------------------------------------
model: # Encoder-Decoder Transformer (HW3P2 in JAX)
  # Speech embedding parameters
  input_dim: 80              # Speech feature dimension
  time_reduction: 2          # Time dimension downsampling factor
  reduction_method: 'conv'   # The source_embedding reduction method ['lstm', 'conv', 'both']

  # Architecture parameters
  d_model: 256           # Model dimension
  num_encoder_layers: 2  # Number of encoder layers
  num_decoder_layers: 2  # Number of decoder layers
  num_encoder_heads: 4   # Number of encoder attention heads
  num_decoder_heads: 4   # Number of decoder attention heads
  d_ff_encoder: 1024     # Feed-forward dimension for encoder
  d_ff_decoder: 1024     # Feed-forward dimension for decoder
  encoder_pos_encoding: 'rope'  # How the encoder receives position ['rope', 'sinusoidal', 'none']
  decoder_pos_encoding: 'rope'  # How the decoder receives position ['rope', 'sinusoidal', 'none']
  norm_type: 'rmsnorm'          # Normalization ['rmsnorm', 'layernorm']
  ffn_type: 'swiglu'            # Feed-forward network ['swiglu', 'gelu']
  qk_norm: False                # Normalize queries and keys before scoring

  # Regularization
  dropout: 0.0          # Dropout rate
  layer_drop_rate: 0.0  # Layer dropout rate
  weight_tying: False   # Whether to use weight tying

###### Training Parameters -------------------------------------------------------
training:
  use_wandb                   : False  # Toggle wandb logging
  wandb_run_id                : "none" # "none" or "run_id"
  resume                      : False  # Resume an existing run (run_id != 'none')
  gradient_accumulation_steps : 1
  wandb_project               : "hw3p2-jax-asr" # wandb project to log to

###### Loss ----------------------------------------------------------------------
loss:
  label_smoothing: 0.0
  ctc_weight: 0.2

###### Optimizer -----------------------------------------------------------------
optimizer:
  name: "adamw" # Options: sgd, adam, adamw
  lr: 0.0004    # Base learning rate
  weight_decay: 0.000001
  adamw:
    betas: [0.9, 0.999]
    eps: 1.0e-8

###### Scheduler -----------------------------------------------------------------
scheduler:
  name: "cosine"  # Options: reduce_lr, cosine, cosine_warm
  cosine:
    T_max: 15
    eta_min: 0.0000001
  warmup:
    enabled: True
    type: "exponential"
    epochs: 5
    start_factor: 0.1
    end_factor: 1.0


In [ ]:
with open('config.yaml', 'r') as file:
    config = yaml.safe_load(file)

## Tokenizer

In [ ]:
Tokenizer = H3Tokenizer(
    token_map  = config['tokenization']['token_map'],
    token_type = config['tokenization']['token_type']
)

## Datasets

In [ ]:
train_dataset = ASRDataset(
    partition=config['data']['train_partition'],
    config=config['data'],
    tokenizer=Tokenizer,
    isTrainPartition=True,
    global_stats=None  # Will compute stats from training data
)

# TODO: Get the computed global stats from training set
global_stats = None
if config['data']['norm'] == 'global_mvn':
    global_stats = (train_dataset.global_mean, train_dataset.global_std)
    print(f"Global stats computed from training set.")

val_dataset = ASRDataset(
    partition=config['data']['val_partition'],
    config=config['data'],
    tokenizer=Tokenizer,
    isTrainPartition=False,
    global_stats=global_stats
)

test_dataset = ASRDataset(
    partition=config['data']['test_partition'],
    config=config['data'],
    tokenizer=Tokenizer,
    isTrainPartition=False,
    global_stats=global_stats
)

gc.collect()

## Dataloaders

In [ ]:
train_loader    = DataLoader(
    dataset     = train_dataset,
    batch_size  = config['data']['batch_size'],
    shuffle     = True,
    num_workers = config['data']['NUM_WORKERS'] if jax.default_backend() == 'gpu' else 0,
    collate_fn  = train_dataset.collate_fn
)

val_loader      = DataLoader(
    dataset     = val_dataset,
    batch_size  = config['data']['batch_size'],
    shuffle     = False,
    num_workers = config['data']['NUM_WORKERS'] if jax.default_backend() == 'gpu' else 0,
    collate_fn  = val_dataset.collate_fn
)

test_loader     = DataLoader(
    dataset     = test_dataset,
    batch_size  = config['data']['batch_size'],
    shuffle     = False,
    num_workers = config['data']['NUM_WORKERS'] if jax.default_backend() == 'gpu' else 0,
    collate_fn  = test_dataset.collate_fn
)

gc.collect()


### Dataloader Verification

In [ ]:
verify_dataloader(train_loader)

In [ ]:
verify_dataloader(val_loader)

In [ ]:
verify_dataloader(test_loader)

## Calculate Max Lengths
Calculating the maximum transcript length across your dataset is a crucial step when working with certain transformer models.
-  We'll use sinusoidal positional encodings that must be precomputed up to a fixed maximum length.
- This maximum length is a hyperparameter that determines:
  - How long of a sequence your model can process
  - The size of your positional encoding matrix
  - Memory requirements during training and inference
- `Requirements`: For this assignment, ensure your positional encodings can accommodate at least the longest sequence in your dataset to prevent truncation. However, you can set this value higher if you anticipate using your languagemodel to work with longer sequences in future tasks (hint: this might be useful for P2! 😉).
- `NOTE`: We'll be using the same positional encoding matrix for all sequences in your dataset. Take this into account when setting your maximum length.

In [ ]:
max_feat_len       = max(train_dataset.feat_max_len, val_dataset.feat_max_len, test_dataset.feat_max_len)
max_transcript_len = max(train_dataset.text_max_len, val_dataset.text_max_len, test_dataset.text_max_len)
max_len            = max(max_feat_len, max_transcript_len)

print("="*50)
print(f"{'Max Feature Length':<30} : {max_feat_len}")
print(f"{'Max Transcript Length':<30} : {max_transcript_len}")
print(f"{'Overall Max Length':<30} : {max_len}")
print("="*50)

## Wandb

In [ ]:
wandb.login(key="your_wandb_api_key_here")

## Training

Every time you run the trainer, it will create a new directory in the `expts` folder with the following structure:
```
expts/
    └── {run_name}/
        ├── config.yaml
        ├── model_arch.txt
        ├── checkpoints/
        │   ├── checkpoint-best-metric-model.pth
        │   └── checkpoint-last-epoch-model.pth
        ├── attn/
        │   └── {attention visualizations}
        └── text/
            └── {generated text outputs}
```


### Training Strategy 1: Cold-Start Trainer

#### Model Load (Default)

In [ ]:
model_config = config['model'].copy()
model_config.update({
    'max_len': max_len,
    'num_classes': Tokenizer.vocab_size
})

rngs = nnx.Rngs(0)
model = EncoderDecoderTransformer(**model_config, rngs=rngs)

# Get sample batch from dataloader
for batch in train_loader:
    padded_feats, padded_shifted, padded_golden, feat_lengths, transcript_lengths = batch
    break

# Count parameters in Flax NNX
graphdef, state = nnx.split(model)
total_param = sum(x.size for x in jax.tree.leaves(state))
print(f"Total model parameters: {total_param:,}")
assert total_param < 30_000_000, f"Total parameters ({total_param}) exceeds 30 million."

# Verify forward pass
logits, ctc_logits, attns = model(padded_feats, padded_shifted, feat_lengths, transcript_lengths)
print(f"Forward pass successful! Logits: {logits.shape}, CTC: {ctc_logits.shape}")


#### Initialize Trainer

If you need to reload the model from a checkpoint, you can do so by calling the `load_checkpoint` method.

```python
checkpoint_path = "path/to/checkpoint.pth"
trainer.load_checkpoint(checkpoint_path)
```


In [ ]:
trainer = ASRTrainer(
    model=model,
    tokenizer=Tokenizer,
    config=config,
    run_name="jax_asr_baseline",
    config_file="config.yaml"
)


### Setup Optimizer and Scheduler

You can set your own optimizer and scheduler by setting the class members in the `ASRTrainer` class.
Eg:
```python
trainer.optimizer = create_optimizer(
    model=model,
    arg2=config['optimizer']
)
```

We also provide a utility function to create your own optimizer and scheduler with the congig and some extra bells and whistles. You are free to use it or not. Do read their code and documentation to understand how it works (`hw3lib/utils/*`).


#### Setting up the optimizer

In [ ]:
trainer.optimizer = create_optimizer(
    model=model,
    arg2=config['optimizer']
)


#### Creating a test scheduler and plotting the learning rate schedule

In [ ]:
test_scheduler = create_scheduler(config['scheduler'])
plot_lr_schedule(test_scheduler, total_steps=1000)


#### Setting up the scheduler

In [ ]:
trainer.scheduler = create_scheduler(config['scheduler'])


#### Train
- Set your epochs and start training!
- `NOTE`: A `scheduler` gets initialized in this call based on the config.

In [ ]:
trainer.train(train_loader, val_loader, epochs=10)

#### Inference



In [ ]:
# Define the recognition config: Greedy search
recognition_config = {
    'num_batches': None,
    'temperature': 1.0,
    'repeat_penalty': 1.0,
    'beam_width': 1, # Beam width of 1 reverts to greedy
}

# Recognize with the config above
config_name = "test"
print(f"Evaluating with {config_name} config")
results = trainer.recognize(test_loader, recognition_config, config_name=config_name, max_length=max_transcript_len)

# Calculate metrics on full batch
generated = [r['generated'] for r in results]
results_df = pd.DataFrame(
    {
        'id': range(len(generated)),
        'transcription': generated
    }
)

# Cleanup
trainer.cleanup()


## Submit to Kaggle

In [ ]:
results_df.to_csv("results.csv", index=False)
api.competition_submit(file_name="results.csv", message="submission for HW3P2", competition="hw-3-p-2-f-26-sandbox")


#### **Manual submission (optional)**

If automatic submission is unavailable, download the CSV and submit it manually using one of the options below.

##### **Kaggle**

* After running the cell above, the file **`results.csv`** will be saved in:
  **Right sidebar → Output → `kaggle/working/`**
* If the file does not appear right away, click the **refresh icon** in the top-right corner of the Output panel.
* Once the file is visible, **right-click → Download**.
* Upload the downloaded file to the **Kaggle competition submission page**.

---

##### **Colab**

```python
from google.colab import files
files.download("results.csv")
```

##### **PSC**

* Open the left file browser.
* Navigate to:
  `/jet/home/<your_username>/`
* Locate **`results.csv`**.
* Right-click the file and select **Download**.
* If the file does not appear immediately, refresh the file browser.

## 📦 Gradescope Submission Instructions (Read Carefully)

This section guides you through creating and submitting your **final Gradescope code submission**.
Please follow each step in order. Skipping steps may result in an incomplete or invalid submission.

### ⏰ Important Deadlines

* **Kaggle Final Deadline:** See the course schedule
* **Gradescope Code Submission Deadline:**
  **48 hours after** the Kaggle deadline (or the same day as your Slack Kaggle submission, if applicable)

You must submit **both**:

1. A valid Kaggle submission
2. A complete Gradescope code submission zip

Please assign your **final trained model** to the global variable `MODEL`.

We will use this variable to automatically generate a file named
`model_metadata.json`, which records important information about your model (such as parameter count and architecture).

#### 🔴 Important Requirements

* **`MODEL` must be the exact model used for your best Kaggle submission**
* This step is **required** for grading and audit purposes
* If the model does not match your Kaggle submission, your score may be invalidated

#### 🕒 When to Do This

* Run this cell **only after you have finished training** your final model
* Do **not** retrain or modify the model after assigning it to `MODEL`


In [ ]:
MODEL = None # TODO: Initialize to your tained model

### Step 2: Complete the README Section

Fill in the `README` variable with a brief description of your work:

* **Model**: Architecture and key design choices
* **Training Strategy**: Optimizer, scheduler, loss, etc.
* **Augmentations**: If used (omit if not)
* **Notebook Execution**: Any instructions needed to run your notebook

This README will be included in your submission zip.

In [ ]:
README = """
- **Model**: Model archtiecture description. Anything unique? Any specific architecture shapes or strategies?
- **Training Strategy**: optimizer + scheduler + loss function + any other unique ideas
- **Augmentations**: augmentations if used. If augmentations weren't used, then ignore
- **Notebook Execution**: Any instructions required to run your notebook.
"""

### Step 3: Provide Required Credentials

Ensure the following variables are correctly set:

* `KAGGLE_USERNAME`
* `KAGGLE_API_KEY`
* `WANDB_API_KEY`
* `WANDB_USERNAME_OR_TEAMNAME`
* `WANDB_PROJECT`

These are used **only** to:

* Fetch your official Kaggle score
* Export your top WandB runs for grading

⚠️ Do **not** share your API keys publicly.

In [ ]:
# @title
KAGGLE_USERNAME = "<your-username>" # TODO
KAGGLE_API_KEY  = "<your-key>"      # TODO
WANDB_API_KEY   = "<your_key>"  # TODO
WANDB_USERNAME_OR_TEAMNAME = "deep-learning-hw-ablations" # TODO: Put your username-or-team-name here
WANDB_PROJECT              = "HW3P2-TA"         # TODO: Put your project-name

### Step 4: Set File Paths Correctly

You must provide absolute path to your **final notebook**

### Platform-specific guidance

**Colab**

* Right-click a file in the left file pane → **Copy path**
* Paths typically start with `/content/...`

**Kaggle**

* Download your notebook (`File → Download Notebook`)
* Upload it via **Upload Input → Upload Model**
* Copy paths from the right sidebar (`/kaggle/working/...`)

**PSC / Linux**

* Locate files under `/jet/home/<your_username>/`
* Use `!ls` to confirm paths

Paste the correct path below.

In [ ]:
NOTEBOOK_PATH = "" # TODO

### Step 5: (Optional) Additional Files

If you have extra files you want to include (e.g., config files, scripts), add their paths to `ADDITIONAL_FILES`, Otherwise, leave this list empty.

In [ ]:
ADDITIONAL_FILES = []

### Step 6: Generate the Final Submission Zip

Before running the submission cell:

* Ensure `ACKNOWLEDGED = True`
* Double-check all paths and credentials
* Confirm your Kaggle submission exists

Then run the cell below.
This should generate a `HW2P2_final_submission.zip` which includes:
* Your notebook
* `model_metadata_*.json`
* README
* WandB run exports
* Kaggle metadata
* Academic integrity acknowledgement

In [ ]:
#### DO NOT MODIFY ####
# TODO: point this at the F26 submission backend once it exists, and import the
# HW3P2 backend config from it. Both the repository name and the config constant
# below are still the S26 ones.
!git clone https://github.com/CMU-IDeeL/S26-HWP2-Submission-Backend.git
!mv S26-HWP2-Submission-Backend/submission .
!rm -rf S26-HWP2-Submission-Backend
from submission.submission_config import SubmissionConfig
from submission.backend_config import BackendConfig, HW4P2_BACKEND_CONFIG
from submission.main import create_submission_zip

create_submission_zip(
    cfg = SubmissionConfig(
        model = 0,
        kaggle_username  = KAGGLE_USERNAME,
        kaggle_api_key   = KAGGLE_API_KEY,
        wandb_api_key    = WANDB_API_KEY,
        wandb_entity     = WANDB_USERNAME_OR_TEAMNAME,
        wandb_project    = WANDB_PROJECT,
        acknowledged     = ACKNOWLEDGED,
        readme           = README,
        notebook_path    = NOTEBOOK_PATH,
        additional_files = ADDITIONAL_FILES
    ),
    backend_cfg = HW4P2_BACKEND_CONFIG
)

### Step 7: Upload to Gradescope

1. Download `HWXP2_final_submission.zip`
2. Upload it to **Gradescope**
3. Verify the upload succeeds

⚠️ Submitting an incomplete zip or modifying backend code may result in grading penalties or an Academic Integrity Violation.

---

### ✅ Final Checklist

Before submitting, confirm:

* [ ] Kaggle username is valid
* [ ] `model_metadata_*.json` matches your best run
* [ ] README is complete
* [ ] File paths are correct
* [ ] `ACKNOWLEDGED = True`
* [ ] Final zip downloads successfully